# Mutation Potentials Demo

Minimal demo of `ProjectedDirectionPairwiseSimilarityPotential` and `compose_mutant_distribution` on a peptide.

In [3]:
import torch
from pep_compass.models.encoder_decoder.hydramp_encoder_decoder import (
    HydrAMPEncoderDecoder,
)
from pep_compass.models.encoder_decoder.utils import decoder_jacobian
from pep_compass.local_enumeration.mutation.utils import get_mutations_from_s_u_standard
from pep_compass.local_enumeration.sampling.sorbes import SubRiemannianTangentSpace

from pep_compass.local_enumeration.mutation.mutation_potentials import (
    compose_mutant_distribution,
    ProjectedDirectionPairwiseSimilarityPotential,
)

In [4]:
device = torch.device("cpu")
hydramp = HydrAMPEncoderDecoder(
    jacobian_mode="approx",
    jacobian_eps=1e-6,
    field_eps=1e-6,
    device=device,
)

In [5]:
peptide = "FLYKWWIRIGRLKL"
pep_len = len(peptide)
z = hydramp.encode_peptides([peptide])
print(f"Parent: {peptide}")
print(f"Latent shape: {z.shape}")

Parent: FLYKWWIRIGRLKL
Latent shape: torch.Size([1, 64])


In [6]:
jac = decoder_jacobian(
    lambda x: hydramp.decoder_forward(x, softmax=True, flatten=True),
    z,
    jacobian_fn_mode="approx",
    jacobian_fn_kwargs={"jacobian_eps": 1e-6},
)
U, S, V = torch.linalg.svd(jac, full_matrices=False)
print(f"U shape: {U.shape}, S shape: {S.shape}, V shape: {V.shape}")

U shape: torch.Size([1, 525, 64]), S shape: torch.Size([1, 64]), V shape: torch.Size([1, 64, 64])


In [7]:
S[0].detach().cpu().numpy()

array([4.0451312e-03, 7.0609455e-04, 3.9391438e-04, 3.7333343e-04,
       2.1011812e-04, 1.6609606e-04, 1.9095794e-05, 1.4202213e-05,
       1.0694105e-05, 9.2550745e-06, 5.8253718e-06, 4.0492087e-06,
       2.9571856e-06, 2.5452664e-06, 2.1410688e-06, 1.6692513e-06,
       1.4492007e-06, 1.3206219e-06, 1.0604336e-06, 9.0314120e-07,
       7.8348864e-07, 4.4403836e-07, 2.9451937e-07, 2.3163933e-07,
       2.0342101e-07, 1.8109969e-07, 1.5192607e-07, 1.3235115e-07,
       1.1537374e-07, 9.6874174e-08, 6.0102366e-08, 5.0703214e-08,
       4.4011024e-08, 3.9289880e-08, 3.7693905e-08, 3.1962770e-08,
       2.9120763e-08, 2.1900409e-08, 1.9686487e-08, 1.7594001e-08,
       1.6995495e-08, 1.1431438e-08, 8.3884650e-09, 5.9861396e-09,
       5.4555698e-09, 5.1401612e-09, 4.3362172e-09, 3.6689869e-09,
       3.4505274e-09, 2.5976643e-09, 2.5482643e-09, 2.1889437e-09,
       1.5051600e-09, 1.2778331e-09, 8.6469443e-10, 7.9915646e-10,
       5.8156147e-10, 4.4114551e-10, 3.6779405e-10, 2.6138544e

In [8]:
mutations = get_mutations_from_s_u_standard(
    s=S[0].detach().cpu().numpy(),
    u=U[0].detach().cpu().numpy(),
    max_len=25,
    alphabet_size=21,
    direction_significance_threshold=1e-4,
    min_number_of_directions=5,
    token_threshold=0.05,
)

alphabet = list(" ACDEFGHIKLMNPQRSTVWY")
print("Proposed mutations:")
for pos in sorted(mutations.keys()):
    aas = [alphabet[i] for i in mutations[pos]]
    print(f"  pos {pos}: indices {mutations[pos]}  ->  {aas}")

Proposed mutations:
  pos 0: indices [20]  ->  ['Y']
  pos 3: indices [19]  ->  ['W']
  pos 6: indices [5, 13, 5, 7, 13]  ->  ['F', 'P', 'F', 'H', 'P']
  pos 8: indices [5, 13]  ->  ['F', 'P']
  pos 12: indices [19]  ->  ['W']


In [9]:
##take only positions od pep_len
mutations = {pos: muts for pos, muts in mutations.items() if pos < pep_len}
print("\nProposed mutations (only positions within peptide length):")
for pos in sorted(mutations.keys()):
    aas = [alphabet[i] for i in mutations[pos]]
    print(f"  pos {pos}: indices {mutations[pos]}  ->  {aas}")


Proposed mutations (only positions within peptide length):
  pos 0: indices [20]  ->  ['Y']
  pos 3: indices [19]  ->  ['W']
  pos 6: indices [5, 13, 5, 7, 13]  ->  ['F', 'P', 'F', 'H', 'P']
  pos 8: indices [5, 13]  ->  ['F', 'P']
  pos 12: indices [19]  ->  ['W']


In [10]:
# Create tangent space for the peptide
tangent_space = SubRiemannianTangentSpace(
    U=U[0], 
    S=S[0], 
    V=V[0],
    horizontal_threshold=1e-4, 
    device="cpu"
)

# Create potential using the new class
potential = ProjectedDirectionPairwiseSimilarityPotential(tangent_space=tangent_space)

df = compose_mutant_distribution(
    parent_peptide=peptide,
    mutations=mutations,
    potential=potential,
    include_parent_residue=True,
    top_k=20,
)

df

,sequence,log_potential
0,FLYWWWFRPGRLWL,-4.932725
1,FLYWWWFRFGRLWL,-4.974263
2,FLYWWWHRPGRLWL,-4.995682
3,FLYWWWHRFGRLWL,-5.042268
4,FLYWWWFRPGRLKL,-5.445107
5,FLYWWWFRFGRLKL,-5.484325
6,FLYKWWFRPGRLWL,-5.563022
7,FLYKWWHRPGRLWL,-5.584459
8,FLYKWWFRFGRLWL,-5.599431
9,FLYKWWHRFGRLWL,-5.625916


In [13]:
potentials_dict = potential.compute(peptide, mutations)
print("Pairwise similarity potentials (Cartesian product format):")
print(f"Total mutant combinations: {len(potentials_dict)}") 

# Check if parent peptide is included (all positions unchanged)
positions = sorted(mutations.keys())
parent_aa_tuple = tuple(alphabet.index(peptide[pos]) for pos in positions)
if parent_aa_tuple in potentials_dict:
    print(f"\n✓ Parent peptide combination IS included: {parent_aa_tuple}")
    print(f"  Score: {potentials_dict[parent_aa_tuple]:.4f}")
else:
    print(f"\n✗ Parent peptide combination NOT found: {parent_aa_tuple}")
    print("  This means include_parent_residue is not working correctly")

# Show top 5 mutant combinations by score
print("\nTop 5 mutant combinations by score:")
sorted_combos = sorted(potentials_dict.items(), key=lambda x: -x[1])[:25]

for aa_tuple, score in sorted_combos:
    mutation_str = ', '.join([f"pos{positions[i]}→{alphabet[aa_idx]}" for i, aa_idx in enumerate(aa_tuple)])
    is_parent = (aa_tuple == parent_aa_tuple)
    marker = " [PARENT]" if is_parent else ""
    print(f"  {mutation_str}: {score:.4f}{marker}")

Pairwise similarity potentials (Cartesian product format):
Total mutant combinations: 6

✗ Parent peptide combination NOT found: (5, 9, 8, 8, 9)
  This means include_parent_residue is not working correctly

Top 5 mutant combinations by score:
  pos0→Y, pos3→W, pos6→H, pos8→P, pos12→W: -5.9852
  pos0→Y, pos3→W, pos6→H, pos8→F, pos12→W: -6.0321
  pos0→Y, pos3→W, pos6→F, pos8→P, pos12→W: -6.0699
  pos0→Y, pos3→W, pos6→F, pos8→F, pos12→W: -6.1118
  pos0→Y, pos3→W, pos6→P, pos8→F, pos12→W: -7.6429
  pos0→Y, pos3→W, pos6→P, pos8→P, pos12→W: -7.6727
